## PakMart Traders – Bronze to Silver

Reads raw CSV files from the Bronze zone (`Files/pakmart/full/`) and writes them as **Delta Lake tables** registered in the Spark catalog.

> **Before running:** Attach the `pakmart` lakehouse as the default lakehouse (Explorer panel → Add data items → Lakehouses → pakmart).

### 0 – Confirm lakehouse is attached

In [ ]:
# Verify the default lakehouse is set correctly before loading any data
print('Current database:', spark.catalog.currentDatabase())
print('Available tables:', [t.name for t in spark.catalog.listTables()])

### 1 – Dimension: City

In [ ]:
from pyspark.sql.types import *

table_name = 'dimension_city'

schema = StructType([
    StructField('CityKey', IntegerType(), True),
    StructField('WWICityID', IntegerType(), True),
    StructField('City', StringType(), True),
    StructField('StateProvince', StringType(), True),
    StructField('Country', StringType(), True),
    StructField('Continent', StringType(), True),
    StructField('SalesTerritory', StringType(), True),
    StructField('Region', StringType(), True),
    StructField('Subregion', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('LatestRecordedPopulation', LongType(), True),
    StructField('ValidFrom', TimestampType(), True),
    StructField('ValidTo', TimestampType(), True),
    StructField('LineageKey', IntegerType(), True)
])

df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'{table_name}: {spark.table(table_name).count()} rows')

### 2 – Dimension: Customer

In [ ]:
from pyspark.sql.types import *

table_name = 'dimension_customer'

schema = StructType([
    StructField('CustomerKey', IntegerType(), True),
    StructField('WWICustomerID', IntegerType(), True),
    StructField('Customer', StringType(), True),
    StructField('BillToCustomer', StringType(), True),
    StructField('Category', StringType(), True),
    StructField('BuyingGroup', StringType(), True),
    StructField('PrimaryContact', StringType(), True),
    StructField('PostalCode', StringType(), True),
    StructField('ValidFrom', TimestampType(), True),
    StructField('ValidTo', TimestampType(), True),
    StructField('LineageKey', IntegerType(), True)
])

df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'{table_name}: {spark.table(table_name).count()} rows')

### 3 – Dimension: Date

Pakistani fiscal year runs **July 1 – June 30**. `FiscalMonthNumber=1` = July.

In [ ]:
from pyspark.sql.types import *

table_name = 'dimension_date'

schema = StructType([
    StructField('Date', TimestampType(), True),
    StructField('DayNumber', IntegerType(), True),
    StructField('Day', StringType(), True),
    StructField('Month', StringType(), True),
    StructField('ShortMonth', StringType(), True),
    StructField('CalendarMonthNumber', IntegerType(), True),
    StructField('CalendarMonthLabel', StringType(), True),
    StructField('CalendarYear', IntegerType(), True),
    StructField('CalendarYearLabel', StringType(), True),
    StructField('FiscalMonthNumber', IntegerType(), True),
    StructField('FiscalMonthLabel', StringType(), True),
    StructField('FiscalYear', IntegerType(), True),
    StructField('FiscalYearLabel', StringType(), True),
    StructField('ISOWeekNumber', IntegerType(), True)
])

df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'{table_name}: {spark.table(table_name).count()} rows')

### 4 – Dimension: Employee

In [ ]:
from pyspark.sql.types import *

table_name = 'dimension_employee'

schema = StructType([
    StructField('EmployeeKey', IntegerType(), True),
    StructField('WWIEmployeeID', IntegerType(), True),
    StructField('Employee', StringType(), True),
    StructField('PreferredName', StringType(), True),
    StructField('IsSalesperson', BooleanType(), True),
    StructField('Photo', StringType(), True),
    StructField('ValidFrom', TimestampType(), True),
    StructField('ValidTo', TimestampType(), True),
    StructField('LineageKey', IntegerType(), True)
])

df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'{table_name}: {spark.table(table_name).count()} rows')

### 5 – Dimension: Stock Item

In [ ]:
from pyspark.sql.types import *

table_name = 'dimension_stock_item'

schema = StructType([
    StructField('StockItemKey', IntegerType(), True),
    StructField('WWIStockItemID', IntegerType(), True),
    StructField('StockItem', StringType(), True),
    StructField('Color', StringType(), True),
    StructField('SellingPackage', StringType(), True),
    StructField('BuyingPackage', StringType(), True),
    StructField('Brand', StringType(), True),
    StructField('Size', StringType(), True),
    StructField('LeadTimeDays', IntegerType(), True),
    StructField('QuantityPerOuter', IntegerType(), True),
    StructField('IsChillerStock', BooleanType(), True),
    StructField('Barcode', StringType(), True),
    StructField('TaxRate', DecimalType(18,3), True),
    StructField('UnitPrice', DecimalType(18,2), True),
    StructField('RecommendedRetailPrice', DecimalType(18,2), True),
    StructField('TypicalWeightPerUnit', DecimalType(18,3), True),
    StructField('Photo', StringType(), True),
    StructField('ValidFrom', TimestampType(), True),
    StructField('ValidTo', TimestampType(), True),
    StructField('LineageKey', IntegerType(), True)
])

df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'{table_name}: {spark.table(table_name).count()} rows')

### 6 – Fact: Sale

Loads all 4 year files at once, derives `Year`, `Quarter`, `Month` columns, and partitions the Delta table by Year/Quarter for fast time-range queries.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, year, month, quarter

table_name = 'fact_sale'

schema = StructType([
    StructField('SaleKey', LongType(), True),
    StructField('CityKey', IntegerType(), True),
    StructField('CustomerKey', IntegerType(), True),
    StructField('BillToCustomerKey', IntegerType(), True),
    StructField('StockItemKey', IntegerType(), True),
    StructField('InvoiceDateKey', TimestampType(), True),
    StructField('DeliveryDateKey', TimestampType(), True),
    StructField('SalespersonKey', IntegerType(), True),
    StructField('WWIInvoiceID', IntegerType(), True),
    StructField('Description', StringType(), True),
    StructField('Package', StringType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('UnitPrice', DecimalType(18,2), True),
    StructField('TaxRate', DecimalType(18,3), True),
    StructField('TotalExcludingTax', DecimalType(29,2), True),
    StructField('TaxAmount', DecimalType(38,6), True),
    StructField('Profit', DecimalType(18,2), True),
    StructField('TotalIncludingTax', DecimalType(38,6), True),
    StructField('TotalDryItems', IntegerType(), True),
    StructField('TotalChillerItems', IntegerType(), True),
    StructField('LineageKey', IntegerType(), True)
])

# Read all year CSV files from the full folder in one pass
df = spark.read.format('csv').schema(schema).option('header', 'true') \
         .load(f'Files/pakmart/full/{table_name}')

# Derive partition columns
df = df.withColumn('Year',    year(col('InvoiceDateKey'))) \
       .withColumn('Quarter', quarter(col('InvoiceDateKey'))) \
       .withColumn('Month',   month(col('InvoiceDateKey')))

# saveAsTable registers in catalog AND writes partitioned Delta files
df.write.mode('overwrite').format('delta') \
        .partitionBy('Year', 'Quarter') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(table_name)

print(f'{table_name}: {spark.table(table_name).count():,} rows')

### 7 – Verify: row counts per partition

In [ ]:
# Python verify — no catalog lookup needed, reads directly from the table object
from pyspark.sql.functions import count as _count

spark.table('fact_sale') \
     .groupBy('Year', 'Quarter', 'Month') \
     .agg(_count('*').alias('Rows')) \
     .orderBy('Year', 'Quarter', 'Month') \
     .show(60)

### 8 – Summary: all tables loaded

In [ ]:
tables = [
    'dimension_city', 'dimension_customer', 'dimension_date',
    'dimension_employee', 'dimension_stock_item', 'fact_sale'
]

print(f'  {"Table":<25}  {"Rows":>10}')
print('  ' + '-' * 37)
total = 0
for t in tables:
    n = spark.table(t).count()
    total += n
    print(f'  {t:<25}  {n:>10,}')
print('  ' + '-' * 37)
print(f'  {"TOTAL":<25}  {total:>10,}')
print()
print('Bronze → Silver complete.')